In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os

# ── Start Spark Session ────────────────────────────────────────────────────
# local[*] means use all CPU cores on this laptop
# No cluster needed — Spark runs locally
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-silver") \
    .getOrCreate()

# Suppress verbose Spark logs — only show errors
spark.sparkContext.setLogLevel("ERROR")

# ── Paths ──────────────────────────────────────────────────────────────────
BRONZE = "/home/jovyan/work/output/bronze"
SILVER = "/home/jovyan/work/output/silver"

os.makedirs(SILVER, exist_ok=True)

print("✅ Spark session started")
print(f"   Spark version : {spark.version}")
print(f"   Bronze path   : {BRONZE}")
print(f"   Silver path   : {SILVER}")

✅ Spark session started
   Spark version : 3.5.0
   Bronze path   : /home/jovyan/work/output/bronze
   Silver path   : /home/jovyan/work/output/silver


In [2]:
# ── Orders Silver ──────────────────────────────────────────────────────────
# What we do here:
# 1. Read the raw bronze CSV
# 2. Cast all date columns from string to proper timestamp type
# 3. Remove exact duplicate rows
# 4. Flag invalid records (delivered before purchase = impossible)

orders_raw = spark.read.csv(f"{BRONZE}/orders.csv", header=True, inferSchema=False)

print(f"Bronze orders: {orders_raw.count()} rows")
print("Columns:", orders_raw.columns)

Bronze orders: 12824 rows
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', '_ingested_at', '_source_endpoint']


In [3]:
# ── Apply cleaning transformations to Orders ───────────────────────────────
orders = (orders_raw
    # Cast all date columns from string to proper timestamp type
    .withColumn("purchase_timestamp",
        F.to_timestamp("order_purchase_timestamp"))
    .withColumn("approved_at",
        F.to_timestamp("order_approved_at"))
    .withColumn("delivered_carrier_date",
        F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("delivered_customer_date",
        F.to_timestamp("order_delivered_customer_date"))
    .withColumn("estimated_delivery_date",
        F.to_timestamp("order_estimated_delivery_date"))

    # Remove exact duplicate rows
    .dropDuplicates(["order_id"])

    # Flag invalid records
    # A delivered date BEFORE purchase date is physically impossible
    .withColumn("_is_valid",
        F.when(
            F.col("delivered_customer_date").isNotNull() &
            (F.col("delivered_customer_date") < F.col("purchase_timestamp")),
            F.lit(False)
        ).otherwise(F.lit(True))
    )
)

# Save to silver
orders.write.mode("overwrite").option("header", True).csv(f"{SILVER}/orders")

# Summary
total = orders.count()
invalid = orders.filter("_is_valid = false").count()
print(f"✅ Orders silver saved")
print(f"   Total rows   : {total}")
print(f"   Invalid rows : {invalid}")
print(f"   Valid rows   : {total - invalid}")

✅ Orders silver saved
   Total rows   : 12824
   Invalid rows : 0
   Valid rows   : 12824


In [4]:
# ── Order Items Silver ─────────────────────────────────────────────────────
# What we do here:
# 1. Cast price and freight_value to decimal numbers
# 2. Cast order_item_id to integer
# 3. Remove duplicates on the natural key (order_id + order_item_id)
# 4. Flag rows with negative prices as invalid

items_raw = spark.read.csv(f"{BRONZE}/order_items.csv", header=True, inferSchema=False)
print(f"Bronze order_items: {items_raw.count()} rows")

items = (items_raw
    .withColumn("order_item_id",
        F.col("order_item_id").cast(IntegerType()))
    .withColumn("price",
        F.col("price").cast(DecimalType(10, 2)))
    .withColumn("freight_value",
        F.col("freight_value").cast(DecimalType(10, 2)))
    .withColumn("shipping_limit_date",
        F.to_timestamp("shipping_limit_date"))

    # Natural key for order items is the combination of order_id + order_item_id
    .dropDuplicates(["order_id", "order_item_id"])

    # Flag negative prices as invalid — price can never be negative
    .withColumn("_is_valid",
        F.when(F.col("price") < 0, F.lit(False))
         .when(F.col("freight_value") < 0, F.lit(False))
         .otherwise(F.lit(True))
    )
)

# Save to silver
items.write.mode("overwrite").option("header", True).csv(f"{SILVER}/order_items")

# Summary
total = items.count()
invalid = items.filter("_is_valid = false").count()
print(f"✅ Order items silver saved")
print(f"   Total rows   : {total}")
print(f"   Invalid rows : {invalid}")
print(f"   Valid rows   : {total - invalid}")

Bronze order_items: 15590 rows
✅ Order items silver saved
   Total rows   : 15590
   Invalid rows : 0
   Valid rows   : 15590


## Validation Tests
The following cell verifies that our data quality checks work correctly
by testing them against intentionally bad synthetic records.
Zero invalid rows in the real data confirms the actual dataset is clean,
not that our checks are missing.

In [5]:
# ── Verify our validation logic actually works ─────────────────────────────
# We will create fake "bad" records and confirm our logic catches them

from pyspark.sql import Row
from datetime import datetime

# ── Test 1: Orders validation ──────────────────────────────────────────────
# Create a fake order where delivered date is BEFORE purchase date
fake_orders = spark.createDataFrame([
    Row(
        order_id="fake_001",
        customer_id="cust_001",
        order_status="delivered",
        order_purchase_timestamp="2018-08-01 10:00:00",
        order_approved_at="2018-08-01 11:00:00",
        order_delivered_carrier_date="2018-08-02 10:00:00",
        order_delivered_customer_date="2018-07-30 10:00:00",  # BEFORE purchase!
        order_estimated_delivery_date="2018-08-10 00:00:00",
        _ingested_at="2026-01-01",
        _source_endpoint="orders"
    )
])

fake_cleaned = (fake_orders
    .withColumn("purchase_timestamp",
        F.to_timestamp("order_purchase_timestamp"))
    .withColumn("delivered_customer_date",
        F.to_timestamp("order_delivered_customer_date"))
    .withColumn("_is_valid",
        F.when(
            F.col("delivered_customer_date").isNotNull() &
            (F.col("delivered_customer_date") < F.col("purchase_timestamp")),
            F.lit(False)
        ).otherwise(F.lit(True))
    )
)

is_valid = fake_cleaned.select("_is_valid").collect()[0][0]
print("── Test 1: Orders validation ──")
print(f"   Fake order with delivery before purchase flagged as invalid: {not is_valid}")
assert not is_valid, "❌ Validation not working for orders!"
print("   ✅ PASSED")

# ── Test 2: Order items validation ────────────────────────────────────────
# Create a fake item with negative price
fake_items = spark.createDataFrame([
    Row(
        order_id="fake_001",
        order_item_id=1,
        product_id="prod_001",
        seller_id="sell_001",
        shipping_limit_date="2018-08-05 00:00:00",
        price="-10.00",   # negative price!
        freight_value="5.00",
        _ingested_at="2026-01-01",
        _source_endpoint="order_items"
    )
])

fake_items_cleaned = (fake_items
    .withColumn("price", F.col("price").cast(DecimalType(10, 2)))
    .withColumn("freight_value", F.col("freight_value").cast(DecimalType(10, 2)))
    .withColumn("_is_valid",
        F.when(F.col("price") < 0, F.lit(False))
         .when(F.col("freight_value") < 0, F.lit(False))
         .otherwise(F.lit(True))
    )
)

is_valid_item = fake_items_cleaned.select("_is_valid").collect()[0][0]
print("\n── Test 2: Order items validation ──")
print(f"   Fake item with negative price flagged as invalid: {not is_valid_item}")
assert not is_valid_item, "❌ Validation not working for order items!"
print("   ✅ PASSED")

print("\n✅ ALL VALIDATION TESTS PASSED")
print("   Zero invalid rows in real data means the actual data is genuinely clean")


── Test 1: Orders validation ──
   Fake order with delivery before purchase flagged as invalid: True
   ✅ PASSED

── Test 2: Order items validation ──
   Fake item with negative price flagged as invalid: True
   ✅ PASSED

✅ ALL VALIDATION TESTS PASSED
   Zero invalid rows in real data means the actual data is genuinely clean


In [7]:
# ── Customers Silver ───────────────────────────────────────────────────────
# What we do here:
# 1. Read raw bronze CSV
# 2. Remove duplicate customer_id rows
# 3. All fields are strings so no type casting needed
# Note: customer_unique_id is the TRUE unique identifier for a customer
# A single person can have multiple customer_ids across orders
# but will always have the same customer_unique_id

customers_raw = spark.read.csv(f"{BRONZE}/customers.csv", header=True, inferSchema=False)
print(f"Bronze customers: {customers_raw.count()} rows")

customers = (customers_raw
    .dropDuplicates(["customer_id"])
    .withColumn("_is_valid", F.lit(True))
)

customers.write.mode("overwrite").option("header", True).csv(f"{SILVER}/customers")

total = customers.count()
unique = customers.select("customer_unique_id").distinct().count()
print(f"✅ Customers silver saved")
print(f"   Total rows          : {total}")
print(f"   Unique customer_ids : {total}")
print(f"   Unique persons      : {unique}")
print(f"   Difference          : {total - unique} (same person, multiple orders)")

Bronze customers: 53441 rows
✅ Customers silver saved
   Total rows          : 53441
   Unique customer_ids : 53441
   Unique persons      : 52395
   Difference          : 1046 (same person, multiple orders)


In [8]:
# ── Products Silver ────────────────────────────────────────────────────────
# What we do here:
# 1. Cast numeric columns to correct types
# 2. Replace null category names with "unknown"
# 3. Remove duplicate product_id rows

products_raw = spark.read.csv(f"{BRONZE}/products.csv", header=True, inferSchema=False)
print(f"Bronze products: {products_raw.count()} rows")

products = (products_raw
    .withColumn("product_weight_g",
        F.col("product_weight_g").cast(DecimalType(10, 2)))
    .withColumn("product_length_cm",
        F.col("product_length_cm").cast(DecimalType(10, 2)))
    .withColumn("product_height_cm",
        F.col("product_height_cm").cast(DecimalType(10, 2)))
    .withColumn("product_width_cm",
        F.col("product_width_cm").cast(DecimalType(10, 2)))
    .withColumn("product_photos_qty",
        F.col("product_photos_qty").cast(IntegerType()))
    .withColumn("product_description_lenght",
        F.col("product_description_lenght").cast(IntegerType()))

    # Replace null category names with "unknown"
    # We never delete data — we fill gaps with a meaningful default
    .withColumn("product_category_name",
        F.when(F.col("product_category_name").isNull(), "unknown")
         .otherwise(F.col("product_category_name"))
    )

    .dropDuplicates(["product_id"])
    .withColumn("_is_valid", F.lit(True))
)

products.write.mode("overwrite").option("header", True).csv(f"{SILVER}/products")

# Check how many had null categories
null_cats = products_raw.filter(F.col("product_category_name").isNull()).count()
print(f"✅ Products silver saved")
print(f"   Total rows             : {products.count()}")
print(f"   Null categories fixed  : {null_cats} (filled with 'unknown')")

Bronze products: 32951 rows
✅ Products silver saved
   Total rows             : 32951
   Null categories fixed  : 610 (filled with 'unknown')


In [9]:
# ── Sellers Silver ─────────────────────────────────────────────────────────
# What we do here:
# 1. Remove duplicate seller_id rows
# 2. All fields are strings so no type casting needed

sellers_raw = spark.read.csv(f"{BRONZE}/sellers.csv", header=True, inferSchema=False)
print(f"Bronze sellers: {sellers_raw.count()} rows")

sellers = (sellers_raw
    .dropDuplicates(["seller_id"])
    .withColumn("_is_valid", F.lit(True))
)

sellers.write.mode("overwrite").option("header", True).csv(f"{SILVER}/sellers")

print(f"✅ Sellers silver saved")
print(f"   Total rows : {sellers.count()}")

Bronze sellers: 3095 rows
✅ Sellers silver saved
   Total rows : 3095


In [10]:
# ── Payments Silver ────────────────────────────────────────────────────────
# What we do here:
# 1. Cast payment_value to decimal
# 2. Cast payment_sequential and payment_installments to integer
# 3. Remove duplicates on order_id + payment_sequential
# Note: One order can have MULTIPLE payment rows
# e.g. someone pays partly by credit card and partly by voucher
# payment_sequential tells us the order of each payment method used

payments_raw = spark.read.csv(f"{BRONZE}/payments.csv", header=True, inferSchema=False)
print(f"Bronze payments: {payments_raw.count()} rows")

payments = (payments_raw
    .withColumn("payment_sequential",
        F.col("payment_sequential").cast(IntegerType()))
    .withColumn("payment_value",
        F.col("payment_value").cast(DecimalType(10, 2)))
    .withColumn("payment_installments",
        F.col("payment_installments").cast(IntegerType()))

    # Natural key is order_id + payment_sequential
    # because one order can have multiple payment rows
    .dropDuplicates(["order_id", "payment_sequential"])
    .withColumn("_is_valid", F.lit(True))
)

payments.write.mode("overwrite").option("header", True).csv(f"{SILVER}/payments")

# Show payment types breakdown
print(f"✅ Payments silver saved")
print(f"   Total rows : {payments.count()}")
print(f"\n   Payment types breakdown:")
payments.groupBy("payment_type") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

Bronze payments: 103886 rows
✅ Payments silver saved
   Total rows : 103886

   Payment types breakdown:
+------------+-----+
|payment_type|count|
+------------+-----+
| credit_card|76795|
|      boleto|19784|
|     voucher| 5775|
|  debit_card| 1529|
| not_defined|    3|
+------------+-----+



In [11]:
# ── Silver Layer Summary ───────────────────────────────────────────────────
# Confirm all silver files were created correctly

import os

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("✅ SILVER LAYER COMPLETE — Summary:")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

silver_tables = {
    "orders"      : orders,
    "order_items" : items,
    "customers"   : customers,
    "products"    : products,
    "sellers"     : sellers,
    "payments"    : payments
}

for name, df in silver_tables.items():
    count = df.count()
    cols  = len(df.columns)
    print(f"   {name:<20} {count:>6} rows, {cols} columns")

print("\n📁 Silver files on disk:")
for folder in os.listdir(SILVER):
    folder_path = f"{SILVER}/{folder}"
    if os.path.isdir(folder_path):
        files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
        print(f"   {folder:<20} {len(files)} CSV part file(s)")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ SILVER LAYER COMPLETE — Summary:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   orders                12824 rows, 16 columns
   order_items           15590 rows, 10 columns
   customers             53441 rows, 8 columns
   products              32951 rows, 12 columns
   sellers                3095 rows, 7 columns
   payments             103886 rows, 8 columns

📁 Silver files on disk:
   orders               1 CSV part file(s)
   customers            4 CSV part file(s)
   payments             5 CSV part file(s)
   products             2 CSV part file(s)
   order_items          1 CSV part file(s)
   sellers              1 CSV part file(s)
